In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy.io as io
import sys
from functools import partial

import src.utils.utils as utils

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(dev)
print(f"Using device: {dev}")

dataset = "urban"
data = io.loadmat(f"datasets/{dataset}.mat")
Y_flat = torch.tensor(data["Y"], dtype=torch.float32)
Y_flat = utils.normalise(Y_flat, dim=0)
E = torch.tensor(data["E"])
B, c, N = E.shape[0], E.shape[1], Y_flat.shape[1]
H = int(N**0.5)

with open(f"/home/ids/edabier/HSU/SS-HSU_benchmark/datasets/{dataset}_wavelength", "r") as file:
    lines = file.readlines()
    wavelengths = [float(line.strip()) for line in lines if line.strip()]

False


In [ ]:
# DOFA

wvs = torch.rand(B)
p = 16
D=128

def get_1d_sincos_pos_embed_from_grid_torch(embed_dim, pos):
    """
    embed_dim: output dimension for each position
    pos: a list of positions to be encoded: size (M,)
    out: (M, D)
    """
    assert embed_dim % 2 == 0
    omega = torch.arange(embed_dim // 2, dtype=torch.float32, device=pos.device)
    omega /= embed_dim / 2.0
    omega = 1.0 / 10000**omega  # (D/2,)

    pos = pos.reshape(-1)  # (M,)
    out = torch.einsum("m,d->md", pos, omega)  # (M, D/2), outer product

    emb_sin = torch.sin(out)  # (M, D/2)
    emb_cos = torch.cos(out)  # (M, D/2)

    emb = torch.cat([emb_sin, emb_cos], dim=1)  # (M, D)
    return emb

class FCResLayer(nn.Module):
    def __init__(self, linear_size=128):
        super(FCResLayer, self).__init__()
        self.l_size = linear_size
        self.nonlin1 = nn.ReLU(inplace=True)
        self.nonlin2 = nn.ReLU(inplace=True)
        self.w1 = nn.Linear(self.l_size, self.l_size)
        self.w2 = nn.Linear(self.l_size, self.l_size)

    def forward(self, x):
        y = self.w1(x)
        y = self.nonlin1(y)
        y = self.w2(y)
        y = self.nonlin2(y)
        out = x + y
        return out

fclayer = FCResLayer(D)
fc_weight = nn.Linear(D, p*p*768)
fc_bias = nn.Linear(D, 768)
wt_num = D
weight_tokens = nn.Parameter(torch.empty([wt_num, D]))
bias_token = nn.Parameter(torch.empty([1, D]))

wvs = get_1d_sincos_pos_embed_from_grid_torch(D, wvs * 1000)
wvs = fclayer(wvs)
x = torch.cat([weight_tokens, wvs], dim=0)
x = torch.cat([x, bias_token], dim=0)
weights = fc_weight(x[wt_num : -1] + wvs)
bias = fc_bias(
    x[-1]
) 

dynamic_weight = weights.view(B, p, p, 768)
dynamic_weight = dynamic_weight.permute([3,0,1,2])
bias = bias.view([768])
in_x = torch.rand(B, 224, 224)
out = F.conv2d(in_x, dynamic_weight, bias=bias, stride=p, padding=1, dilation=1)
print(out.shape)

In [2]:
checkpoint, image_size, vit_patch_size, encoder_global_attn_indexes=None, H, 16, [5, 8, 11]
merge_indexs, class_number, encoder_embed_dim, encoder_depth, encoder_num_heads=[3, 12], -1, 768, 12, 12
prompt_embed_dim, mlp_ratio, qkv_bias, use_rel_pos, window_size, rel_pos_zero_init = 256, 4, True, True, 14, True
image_embedding_size = image_size // vit_patch_size

spectral_wavelength = [400, 412.5, 429.5, 443, 455, 467.5, 473.375, 481.25, 488.25, 
                       500, 520, 531, 536, 545, 550.5, 561.25, 564.75, 565.5, 575, 580, 
                       596, 605, 610, 612, 626, 627.5, 630, 635, 640, 645, 650, 655, 656, 
                       660, 664.5, 665, 667, 671.25, 677.5, 686, 700, 705, 710, 716, 725, 
                       730, 740, 748.5, 760, 764.25, 776, 783, 790, 808, 820, 825, 830, 
                       835.3125, 842, 850, 858.5, 865, 866, 869.5, 880, 896, 905, 910, 926, 
                       938, 945, 950, 959, 1240, 1375, 1575, 1575.5, 1610, 1640, 1650, 2050.25, 
                       2130, 2195, 2217.5,2500]
weight_bank_wavelength = np.arange(400,2510,10).tolist()

x = torch.rand(1, B, image_size, image_size)

In [ ]:
sys.path.append("/home/ids/edabier/HSU/HyperFree")
from HyperFree.HyperFree.modeling.image_encoder import Block, PatchMerging
from HyperFree.HyperFree.modeling.scale_aware_PE import get_2d_sincos_pos_embed_with_resolution
from HyperFree.HyperFree.modeling.common import LayerNorm2d
from HyperFree.HyperFree.utils.spectral_process_utils import find_corresponding_indices, generate_random_indices

class MLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        output_dim: int,
        num_layers: int,
        sigmoid_output: bool = False,
    ) -> None:
        super().__init__()
        self.num_layers = num_layers
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(
            nn.Linear(n, k) for n, k in zip([input_dim] + h, h + [output_dim])
        )
        self.sigmoid_output = sigmoid_output

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < self.num_layers - 1 else layer(x)
        if self.sigmoid_output:
            x = F.sigmoid(x)
        return x

pos_embed_mlp = MLP(encoder_embed_dim, encoder_embed_dim//2, encoder_embed_dim, 3, sigmoid_output=False)
norm_layer = partial(torch.nn.LayerNorm, eps=1e-6)
act_layer = nn.GELU
out_chans = prompt_embed_dim

blocks = nn.ModuleList()
for i in range(encoder_depth):
    block = Block(
        dim=encoder_embed_dim,
        num_heads=encoder_num_heads,
        mlp_ratio=mlp_ratio,
        qkv_bias=qkv_bias,
        norm_layer=norm_layer,
        act_layer=act_layer,
        use_rel_pos=use_rel_pos,
        rel_pos_zero_init=rel_pos_zero_init,
        window_size=window_size if i not in encoder_global_attn_indexes else 0,
        input_size=(image_size // vit_patch_size, image_size // vit_patch_size),
    )
    blocks.append(block)

contras_modules = nn.ModuleList()
for i in range(2):
    block = Block(
        dim=256,
        num_heads=8,
        mlp_ratio=mlp_ratio,
        qkv_bias=qkv_bias,
        norm_layer=norm_layer,
        act_layer=act_layer,
        use_rel_pos=use_rel_pos,
        rel_pos_zero_init=rel_pos_zero_init,
        window_size=16,
        input_size=(image_size // vit_patch_size, image_size // vit_patch_size),
    )
    contras_modules.append(block)

neck = nn.Sequential(
    nn.Conv2d(
        encoder_embed_dim,
        out_chans,
        kernel_size=1,
        bias=False,
    ),
    LayerNorm2d(out_chans),
    nn.Conv2d(
        out_chans,
        out_chans,
        kernel_size=3,
        padding=1,
        bias=False,
    ),
    LayerNorm2d(out_chans),
)

def find_indices_not_in_A(self, A, B):
    set_A = set(A)
    result_indices = []
    for index, element in enumerate(B):
        if element not in set_A:
            result_indices.append(index)
    return result_indices

nm_dis = 10
input_wavelengths_hy = wavelengths
Band_feature_indices_hy, unmatch_indices_hy, point_bank_indices_hy = find_corresponding_indices(input_wavelengths_hy, spectral_wavelength,nm_dis)
weight_bank_data_indices_hy, _, weight_bank_indices_hy = find_corresponding_indices(input_wavelengths_hy, weight_bank_wavelength,nm_dis)

point_spectral_weight_bank_w = nn.Parameter(torch.randn((encoder_embed_dim, len(spectral_wavelength), vit_patch_size, vit_patch_size)))
point_spectral_weight_bank_b = nn.Parameter(torch.randn(encoder_embed_dim))
block_spectral_weight_bank_w = nn.Parameter(torch.randn((encoder_embed_dim, len(weight_bank_wavelength), vit_patch_size, vit_patch_size)))
block_spectral_weight_bank_b = nn.Parameter(torch.randn(encoder_embed_dim))

merge_indexs = merge_indexs
global_attn_indexes = encoder_global_attn_indexes
multi_scale_convs = nn.ModuleList([
    PatchMerging(dim=encoder_embed_dim)
    for i in range(len(merge_indexs))]) if merge_indexs != None else None

In [ ]:
input_wavelengths_hy = wavelengths
Band_feature_indices_hy, unmatch_indices_hy, point_bank_indices_hy = find_corresponding_indices(input_wavelengths_hy, spectral_wavelength,nm_dis)
weight_bank_data_indices_hy, _, weight_bank_indices_hy = find_corresponding_indices(input_wavelengths_hy, weight_bank_wavelength,nm_dis)

random_indices = generate_random_indices(len(weight_bank_data_indices_hy)-1, 40)
random_indices.sort()
indices = [Band_feature_indices_hy, point_bank_indices_hy,  np.array(weight_bank_data_indices_hy)[random_indices].tolist(), np.array(weight_bank_indices_hy)[random_indices].tolist()]

block_indices = find_indices_not_in_A(indices[0], indices[2])
indices[2] = np.array(indices[2])[block_indices].tolist()
indices[3] = np.array(indices[3])[block_indices].tolist()
last_indices = indices

GSD = [1.0]

point_feature = F.conv2d(
x[:,indices[0],:,:],
weight=point_spectral_weight_bank_w[:,indices[1],:,:],
bias=point_spectral_weight_bank_b,
stride=(vit_patch_size, vit_patch_size),
padding=(0,0)
)

if len(indices[2]) > 0:
    block_feature = F.conv2d(
    x[:,indices[2],:,:],
    weight=block_spectral_weight_bank_w[:,indices[3],:,:],
    bias=block_spectral_weight_bank_b,
    stride=(vit_patch_size, vit_patch_size),
    )

if len(indices[2]) > 0:
    x_feature = point_feature + block_feature
else:
    x_feature = point_feature

scale_aware_pos_embed = get_2d_sincos_pos_embed_with_resolution(embed_dim, int(img_size/vit_patch_size), GSD, device=x.device)
scale_aware_pos_embed = pos_embed_mlp(scale_aware_pos_embed)
scale_aware_pos_embed = scale_aware_pos_embed.reshape((x.shape[0], int(img_size/vit_patch_size), int(img_size/vit_patch_size), embed_dim))

x_feature = x_feature.permute((0,2,3,1))
x_feature = x_feature + scale_aware_pos_embed

multi_stage_features = []

multi_scale_merge_index = 0
for i, blk in enumerate(blocks):
    x_feature = blk(x_feature)

    if merge_indexs != None:
        if i in [merge_indexs[0], global_attn_indexes[0], global_attn_indexes[2]]:
            multi_stage_features.append(x_feature.permute(0, 3, 1, 2))

        if i in merge_indexs:
            x_feature = multi_scale_convs[multi_scale_merge_index](x_feature)
            multi_scale_merge_index += 1
    elif i in global_attn_indexes:
        multi_stage_features.append(x_feature.permute(0, 3, 1, 2))

x_feature = neck(x_feature.permute(0, 3, 1, 2))
multi_stage_features.append(x_feature)

multi_stage_features

In [ ]:
sys.path.append("/home/ids/edabier/HSU/HyperFree")
from HyperFree.HyperFree.modeling import ImageEncoderViT, MaskDecoder, PromptEncoder, Sam, TwoWayTransformer


hyperfree = Sam(
    image_encoder=ImageEncoderViT(
        depth=encoder_depth,
        embed_dim=encoder_embed_dim,
        img_size=image_size,
        mlp_ratio=4,
        norm_layer=partial(torch.nn.LayerNorm, eps=1e-6),
        num_heads=encoder_num_heads,
        patch_size=vit_patch_size,
        qkv_bias=True,
        use_rel_pos=True,
        global_attn_indexes=encoder_global_attn_indexes,
        merge_indexs = merge_indexs,
        window_size=14,
        out_chans=prompt_embed_dim,
    ),
    prompt_encoder=PromptEncoder(
        embed_dim=prompt_embed_dim,
        image_embedding_size=(image_embedding_size, image_embedding_size),
        input_image_size=(image_size, image_size),
        mask_in_chans=16,
    ),
    mask_decoder=MaskDecoder(
        num_multimask_outputs=3,
        transformer=TwoWayTransformer(
            depth=2,
            embedding_dim=prompt_embed_dim,
            mlp_dim=2048,
            num_heads=8,
        ),
        transformer_dim=prompt_embed_dim,
        iou_head_depth=3,
        iou_head_hidden_dim=256,
        class_number = class_number
    ),
)